# 사전 준비

In [1]:
import warnings
warnings.filterwarnings('ignore')
import re, os, json, uuid
from textwrap import dedent
from pprint import pprint

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from typing import List, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.tools import TavilySearchResults
from langgraph.graph import START, END, StateGraph, MessagesState
from IPython.display import Image, display

# 다양한 에이전트 구현 준비 - Adaptive RAG, Self RAG, Corrective RAG

## 메뉴 검색을 위한 restaurant_menu 벡터저장소를 로드해서 search_menu라는 도구를 만든다.

벡터저장소를 만들 때 사용한 임베딩 모델을 설정하고 벡터저장소를 로드한다.

In [5]:
# 임베딩 모델 설정
embeddings_model = OllamaEmbeddings(model='bge-m3')

# 벡터저장소를 로드
menu_db = Chroma(
    embedding_function=embeddings_model,
    collection_name='restaurant_menu',
    persist_directory='./chroma_db'
)
print(menu_db._collection.count())

10


@tool 테코레이터를 이용해서 search_menu 도구를 만든다.

In [6]:
@tool
def search_menu(query: str) -> List[Document]:
    # @tool 데코레이터를 사용하면 첫 줄에 반드시 docstring을 넣어야 한다.
    '''벡터저장소의 restaurant_menu에서 메뉴 정보를 검색합니다. 이 도구는 메뉴 관련 조회에만 사용하십시오.'''
    docs = menu_db.similarity_search(query=query, k=2)
    # 검색된 문서(Document 객체)를 리턴시킨다.
    if len(docs) > 0:
        return docs
    return [Document(page_content='벡터저장소에서 메뉴 정보를 찾을 수 없습니다.')]

## 와인 검색을 위한 restaurant_wine 벡터저장소를 로드해서 search_wine라는 도구를 만든다.

벡터저장소를 만들 때 사용한 임베딩 모델을 설정하고 벡터저장소를 로드한다.

In [7]:
# 임베딩 모델 설정
# embeddings_model = OllamaEmbeddings(model='bge-m3')

# 벡터저장소를 로드
wine_db = Chroma(
    embedding_function=embeddings_model,
    collection_name='restaurant_wine',
    persist_directory='./chroma_db'
)
print(wine_db._collection.count())

10


@tool 테코레이터를 이용해서 search_wine 도구를 만든다.

In [8]:
@tool
def search_wine(query: str) -> List[Document]:
    # @tool 데코레이터를 사용하면 첫 줄에 반드시 docstring을 넣어야 한다.
    '''벡터저장소의 restaurant_wine에서 와인 정보를 검색합니다. 이 도구는 와인 관련 조회에만 사용하십시오.'''
    docs = menu_db.similarity_search(query=query, k=2)
    # 검색된 문서(Document 객체)를 리턴시킨다.
    if len(docs) > 0:
        return docs
    return [Document(page_content='벡터저장소에서 와인 정보를 찾을 수 없습니다.')]

## 웹 검색을 위한 해서 Tavily API를 이용해서 search_web이라는 도구를 만든다.

In [9]:
@tool
def search_web(query: str) -> List[Document]:
    # @tool 데코레이터를 사용하면 첫 줄에 반드시 docstring을 넣어야 한다.
    '''search_menu 도구와 search_wine 도구에 존재하지 않는 정보나 최신 정보는 인터넷에서 검색합니다.'''
    # Tavily API 웹 검색 도구를 만들고 질문을 전달해서 실제로 인터넷 검색을 수행하고 결과를 받아온다.
    tavily_search = TavilySearchResults(max_results=2)
    docs = tavily_search.invoke(query)
    
    # 검색된 문서(딕셔너리)를 하나씩 반복하며 Document 객체가 저장된 리스트로 만들어 리턴시킨다.
    results = []
    for doc in docs:
        results.append(
            Document(page_content=doc['content'], metadate={'source': doc['url']})
        )
    
    # 검색된 문서(Document 객체)를 리턴시킨다.
    if len(results) > 0:
        return results
    return [Document(page_content='웹에서 관련 정보를 찾을 수 없습니다.')]

## 도구 목록 통합

지금까지 정의한 3개의 도구(search_menu, search_wine, search_web) 리스트로 묶어준다.

In [10]:
tools = [search_menu, search_wine, search_web]

## LLM 모델을 만들고 도구 목록을 결합한다.

In [13]:
# LLM 모델을 만든다.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, streaming=True)

# LLM 모델에 도구를 결합한다.
llm_with_tools = llm.bind_tools(tools=tools)

## 질문에 따른 도구 호출 여부를 확인한다.

In [15]:
query = '안녕하세요'
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_01a55b9f25', 'service_tier': 'default'}, id='run--01a157f0-a224-7d52-8b6f-a2c089d72a8e-0', usage_metadata={'input_tokens': 149, 'output_tokens': 11, 'total_tokens': 160, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


In [16]:
query = '스테이크 메뉴의 가격은 얼마인가요?'
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_imJr8f2AR6THAg4oaMJPktrF', 'function': {'arguments': '{"query":"스테이크"}', 'name': 'search_menu'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_01a55b9f25', 'service_tier': 'default'}, id='run--01a157f1-c2cc-72b3-a370-11a222bba96f-0', tool_calls=[{'name': 'search_menu', 'args': {'query': '스테이크'}, 'id': 'call_imJr8f2AR6THAg4oaMJPktrF', 'type': 'tool_call'}], usage_metadata={'input_tokens': 158, 'output_tokens': 16, 'total_tokens': 174, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


In [17]:
query = '스테이크에 어울리는 와인을 추천해주세요.'
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_aXKReQHOgfQUi0adL801t4no', 'function': {'arguments': '{"query":"스테이크"}', 'name': 'search_wine'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_01a55b9f25', 'service_tier': 'default'}, id='run--01a157f2-a0e5-7893-81b3-1aa02c3e1a70-0', tool_calls=[{'name': 'search_wine', 'args': {'query': '스테이크'}, 'id': 'call_aXKReQHOgfQUi0adL801t4no', 'type': 'tool_call'}], usage_metadata={'input_tokens': 160, 'output_tokens': 17, 'total_tokens': 177, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


In [18]:
query = 'LangGraph에 대해서 알려주세요.'
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_ZkCUH8nUIkyussoCB55Rqz31', 'function': {'arguments': '{"query":"LangGraph"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_01a55b9f25', 'service_tier': 'default'}, id='run--01a157f3-72e7-7fd0-b05f-42a27b0b7c03-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'LangGraph'}, 'id': 'call_ZkCUH8nUIkyussoCB55Rqz31', 'type': 'tool_call'}], usage_metadata={'input_tokens': 154, 'output_tokens': 15, 'total_tokens': 169, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


In [19]:
query = '채식주의자를 위한 메뉴가 있나요? 채식주의자 메뉴에 어울리는 와인을 추천해주세요.'
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_UTsOJ6VizLHnB7CBLYeiszZG', 'function': {'arguments': '{"query": "채식주의자"}', 'name': 'search_menu'}, 'type': 'function'}, {'index': 1, 'id': 'call_ndjRqbmt9HfpguI8bvZe6gw9', 'function': {'arguments': '{"query": "채식주의자"}', 'name': 'search_wine'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_01a55b9f25', 'service_tier': 'default'}, id='run--01a157f4-d026-7893-9b5c-1709f561ce01-0', tool_calls=[{'name': 'search_menu', 'args': {'query': '채식주의자'}, 'id': 'call_UTsOJ6VizLHnB7CBLYeiszZG', 'type': 'tool_call'}, {'name': 'search_wine', 'args': {'query': '채식주의자'}, 'id': 'call_ndjRqbmt9HfpguI8bvZe6gw9', 'type': 'tool_call'}], usage_metadata={'input_tokens': 171, 'output_tokens': 51, 'total_tokens': 222, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


In [20]:
query = '파스타 메뉴가 있나요? 파스타에 어울리는 와인을 추천해주세요. 그리고 경기도 성남시의 파스타 맛집을 추천해주세요.'
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)

AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_uCjbzfdfmBivZdpts2WqjT8J', 'function': {'arguments': '{"query": "파스타"}', 'name': 'search_menu'}, 'type': 'function'}, {'index': 1, 'id': 'call_0pfaSKFkrkNNbGO4GS01bMPX', 'function': {'arguments': '{"query": "파스타"}', 'name': 'search_wine'}, 'type': 'function'}, {'index': 2, 'id': 'call_AChJgXQs9OplBtr9zNsF66gj', 'function': {'arguments': '{"query": "경기도 성남시 파스타 맛집"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_01a55b9f25', 'service_tier': 'default'}, id='run--01a157f5-c1a0-74a1-be9b-ce2fad29da67-0', tool_calls=[{'name': 'search_menu', 'args': {'query': '파스타'}, 'id': 'call_uCjbzfdfmBivZdpts2WqjT8J', 'type': 'tool_call'}, {'name': 'search_wine', 'args': {'query': '파스타'}, 'id': 'call_0pfaSKFkrkNNbGO4GS01bMPX', 'type': 'tool_call'}, {'name': 'search_web', 'args': {'query': '경기도 성남시 파스타 맛집'}, 'i

# Adaptive(적응형) RAG

질문의 복잡성에 따라 가장 적합한 검색 및 생성 전략을 동적으로 선택하는 방법으로 기존의 전통적인 RAG는 질문이 단순하든 복잡하든 무조건 문서를 검색하고 LLM에 전달하기 때문에, 불필요한 검색으로 인해 속도가 느려지거나 엉뚱한 문서가 섞여 오히려 답변 품질이 떨어지는 문제가 있었다. 어댑티브 RAG는 이 문제를 해결하기 위해 라우팅(Routing) 개념을 도입했다.

<img src="./adaptiveRAG.png" width="1200" align="left" />

# Adaptive RAG 에이전트 구현하기

<img src="./adaptiveRAG2.png" width="1200" align="left" />

동작 방식  
① 질문 입력: 사용자가 질문을 입력한다.  
② 질문 분석: 질문 분류기가 질문을 분석한다.  
③ 전략 선택: 분석 결과에 따라 가장 적합한 처리 전략을 선택한다.  
④ 처리 및 응답: 선택된 전략에 따라 질문을 처리하고 응답을 생성한다.

## 상태(state, schema)를 정의한다.

TypedDict 클래스를 상속받아 LangGraph의 노드들이 공유해서 상태로 사용할 클래스를 만든다. 이 클래스는 일반 클래스가 아니라, 엄격한 규칙을 가진 딕셔너리가 된다.

In [22]:
class AdaptiveRagState(TypedDict):
    # 사용자가 입력한 질문이나 프롬프트를 저장할 key
    question: str
    # 검색 엔진이나 벡터저장소에서 찾아온 관련 문서들을 보관할 key
    documents: List[Document]
    # 질문과 검색된 문서를 바탕으로 AI가 최종적으로 만들어낸 답변을 저장할 key
    answer: str

LangGraph의 모든 노드들이 {'question': '질문문자열', 'documents': '[검색된문서들]', 'answer': '답변문자열'} 형태의 상태를 사용한다.

In [23]:
AdaptiveRagState.__annotations__.keys()

dict_keys(['question', 'documents', 'answer'])

## 질문을 분석한다. => 라우팅

사용자의 질문을 분석해서 적절한(메뉴, 와인, 웹) 도구를 선택한다.

`Literal`: 함수의 리턴 값이나 변수에 저장되는 값이 `지정된 후보군`만 가질 수 있도록 제한한다.  
`BaseModel`: 데이터의 구조(스키마)를 정의한다.  
`Field`: BaseModel 클래스를 상속받아 만든 데이터의 구조의 각 필드의 자료형, 필수 여부, 설명을 설정한다.

In [24]:
from typing import Literal
from pydantic import BaseModel, Field

LLM이 최종적으로 응답할 데이터 구조를 BaseModel 클래스를 상속받아 정의한다.

In [29]:
class ToolSelection(BaseModel):
    # docstring
    # LangChain은 이 설명을 읽어서 이 모델이 어떤 역할을 하는지 LLM에게 전달한다.
    '''사용자의 질문을 처리하기에 가장 알맞은 도구를 선택합니다.'''
    # tool이라는 필드는 오직 'search_menu', 'search_wine', 'search_web' 이 세가지 문자열만 필드에 저장될 수 있다고 제한한다.
    # '...'은 필수라는 의미이고 description은 LLM에게 주는 힌트이다.
    tool: Literal['search_menu', 'search_wine', 'search_web'] = Field(
        description='사용자의 질문을 바탕으로 분석해서 search_menu, search_wine, search_web 도구 중 하나를 선택하세요.'
    )

In [30]:
print(json.dumps(ToolSelection.model_json_schema(), indent=2, ensure_ascii=False))

{
  "description": "사용자의 질문을 처리하기에 가장 알맞은 도구를 선택합니다.",
  "properties": {
    "tool": {
      "description": "사용자의 질문을 바탕으로 분석해서 search_menu, search_wine, search_web 도구 중 하나를 선택하세요.",
      "enum": [
        "search_menu",
        "search_wine",
        "search_web"
      ],
      "title": "Tool",
      "type": "string"
    }
  },
  "required": [
    "tool"
  ],
  "title": "ToolSelection",
  "type": "object"
}


구조화된 출력을 위한 LLM을 설정한다.

기존의 llm 객체에 with_structured_output() 메소드를 이용해서 LLM이 최종적으로 응답할 데이터 구조를 넘겨주면 llm 객체에게 마음대로 답하지 말고 방금 지정한 데이터 구조인 ToolSelection 형식에 맞춰서 대답하라는 의미이다.

In [31]:
structured_llm = llm.with_structured_output(ToolSelection)

프롬프트를 만든다.

In [42]:
system = dedent('''
    당신은 사용자의 질문을 가장 적절한 도구로 연결(라우팅)하는 역할을 전담하는 AI 어시스턴트입니다.
    아래의 지침에 따라 행동하세요.
        - 레스토랑 메뉴에 관한 질문은 search_menu 도구를 사용하세요.
        - 와인 추천이나 페어링(음식 궁합)에 관한 질문은 search_wine 도구를 사용하세요.
        - 그 외의 다른 정보나 최신 정보가 필요한 경우에는 search_web 도구를 사용하세요.
    언제나 사용자의 질문을 바탕으로 가장 적절한 도구를 하나만 선택해야 합니다.
''')

prompt = ChatPromptTemplate.from_messages([
    ('system', system),
    ('human', '{question}')
])

질문 분석 체인을 만든다.

In [43]:
rag_chain = prompt | structured_llm

질문 분석 결과에 따른 도구 호출 테스트

In [46]:
pprint(rag_chain.invoke({'question': '채식주의자를 위한 메뉴가 있나요?'}))
pprint(rag_chain.invoke({'question': '스테이크 메뉴와 어울리는 와인을 추천해주세요.'}))
pprint(rag_chain.invoke({'question': '2026년 월드컵 우승 국가는 어디인가요?'}).tool)

ToolSelection(tool='search_menu')
ToolSelection(tool='search_wine')
'search_web'


여기까지 만든 코드는 사용자가 질문을 던지만 질문 내용에 딱 맞는 도구 하나만 골라는 AI 라우팅 시스템이다.

## AI 라우팅 시스템을 이용해서 질문 라우팅 노드를 만든다.

LangGraph가 공유하는 상태를 받아서 위에서 만든 AI 라우팅 시스템을 실행하고 'search_menu', 'search_wine', 'search_web', 'llm_fallback' 중에 하나를 리턴한다.

In [ ]:
def route_question_node(state: AdaptiveRagState) -> Literal['search_menu', 'search_wine', 'search_web', 'llm_fallback']:
    # 상태에서 질문을 꺼내온다.
    
    # 질문으로 AI 라우팅 시스템을 실행한다.
    
    # AI 라우팅 시스템을 실행 결과에 따라서 실행할 노드 이름을 리턴한다.
    